# SHAP-IMV — Titanic

Exact Shapley attribution with IMV as the coalition value, on the OpenML Titanic
dataset (binary: survived).

`Title` is extracted from the passenger name, so this specification includes the
eight-feature set used in the published figure. Note that `Title` is almost
perfectly nested inside `Sex`, and Shapley symmetry makes near-redundant features
share credit — expect the two to split attribution.

**Pipeline**: download from OpenML → preprocess → standardise → 10 seeds × 3
estimators × exact power set → aggregate.

Full documentation: `documentation/examples/shap_imv/titanic.md`.

In [ ]:
import os, sys, tempfile, warnings, itertools
from pathlib import Path

_PATH_DISPLAY_ROOT = Path()

def relative_path(path, *, start=None):
    """Return a display-only path relative to an explicit local anchor."""
    anchor = Path(start) if start is not None else _PATH_DISPLAY_ROOT
    try:
        relative = os.path.relpath(Path(path).expanduser().resolve(),
                                   start=anchor.expanduser().resolve())
    except ValueError:  # Paths on different Windows drives cannot be relativized.
        relative = Path(path).name
    return Path(relative).as_posix()

def _relative_warning_text(message):
    text = str(message)
    roots = {Path.home(), Path(sys.prefix), Path(sys.base_prefix),
             _PATH_DISPLAY_ROOT, Path(tempfile.gettempdir())}
    for root in sorted(roots, key=lambda value: len(str(value)), reverse=True):
        absolute = str(root.expanduser().resolve())
        text = text.replace(absolute, relative_path(absolute))
    return text

def _show_relative_warning(message, category, filename, lineno, file=None, line=None):
    stream = file if file is not None else sys.stderr
    warning_text = _relative_warning_text(message)
    print(f"{relative_path(filename)}:{lineno}: {category.__name__}: {warning_text}", file=stream)

warnings.showwarning = _show_relative_warning

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# No dataset is stored in this repository. Everything downloads on demand into a
# user-level cache outside the working tree.
CACHE = Path(os.environ.get("IMV_CACHE_HOME", Path.home() / ".cache" / "imv"))
CACHE.mkdir(parents=True, exist_ok=True)

DATA_HOME = Path(os.environ.get("IMV_DATA_CACHE", CACHE / "datasets")) / "openml"
ARTIFACTS = Path(os.environ.get("IMV_ARTIFACT_CACHE", CACHE / "notebook_artifacts")) / "shap_imv_titanic"
RESULTS = ARTIFACTS / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES = ARTIFACTS / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)

# Ten seeds, as required for any reported IMV result. The seed drives the fold
# split and the estimator, so every number below is a mean over ten independent
# fold partitions.
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
N_SPLITS = 5

import imvpy
from imvpy import BinaryIMV
from imvpy.utils import save_figure

IMVPY_SOURCE = Path(imvpy.__file__).resolve().parent
print(f"imvpy {imvpy.__version__} from {relative_path(IMVPY_SOURCE)}")


## 1. Download

Nothing is read from the repository.

In [ ]:
from sklearn.datasets import fetch_openml
DATASET, TITLE = "titanic", "Titanic"
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    frame = fetch_openml("titanic", version=1, as_frame=True,
                         data_home=DATA_HOME).frame.copy()
print(frame.shape, list(frame.columns))
frame.head()

## 2. Preprocess

Every transformation is applied here and stated in the docs.

In [ ]:
frame["target"] = frame["survived"].astype(int)
frame["Sex"] = (frame["sex"].astype(str) == "female").astype(int)
frame["Class"] = pd.to_numeric(frame["pclass"], errors="coerce")
frame["Age"] = pd.to_numeric(frame["age"], errors="coerce")
frame["Age"] = frame["Age"].fillna(frame["Age"].median())
frame["Fare"] = pd.to_numeric(frame["fare"], errors="coerce")
frame["Fare"] = frame["Fare"].fillna(frame["Fare"].median())
frame["Alone"] = ((pd.to_numeric(frame["sibsp"], errors="coerce").fillna(0)
                   + pd.to_numeric(frame["parch"], errors="coerce").fillna(0)) == 0).astype(int)
frame["Embarked"] = frame["embarked"].astype(str).map({"C": 0, "Q": 1, "S": 2}).fillna(2)
frame["AgeClass"] = frame["Age"] * frame["Class"]

# Title from the name field. The published figure's encoding was never recorded,
# so this ordering is our own and is documented rather than inferred.
title = frame["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
title = title.replace(["Lady","Countess","Capt","Col","Don","Dr","Major","Rev",
                       "Sir","Jonkheer","Dona"], "Rare")
title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
frame["Title"] = title.map({"Mr":1,"Miss":2,"Mrs":3,"Master":4,"Rare":5}).fillna(0).astype(int)

FEATURES = ["Sex","Title","Class","AgeClass","Fare","Embarked"]
data = frame[FEATURES + ["target"]].apply(pd.to_numeric, errors="coerce").dropna()
print(data.shape, "positive rate:", round(data["target"].mean(), 4))
print("Title vs Sex (redundancy check):")
print(pd.crosstab(data["Title"], data["Sex"]))
data.head()

## 3. Estimators and budget

In [ ]:
# Exact SHAP-IMV enumerates every feature subset, so cost is
# 2**n_features * n_splits * 2 fits per seed per estimator. That total is driven by
# the coalition count, not the row count, so every example runs on the full dataset
# and the budget is spent on parallelism instead: N_JOBS spreads the 2**n
# coalitions across cores.
N_JOBS = 11

def model_factories(seed):
    """Three estimator families, deliberately small so the exact power set is affordable."""
    return {
        "logistic_regression": lambda: LogisticRegression(max_iter=5000, random_state=seed),
        "xgboost": lambda: XGBClassifier(
            n_estimators=60, max_depth=3, learning_rate=0.2, random_state=seed,
            verbosity=0, tree_method="hist", n_jobs=1),
        "lightgbm": lambda: LGBMClassifier(
            n_estimators=60, max_depth=3, learning_rate=0.2, random_state=seed,
            verbose=-1, n_jobs=1),
    }


## 4. Exact SHAP-IMV over ten seeds

Each value is a mean over ten independent fold partitions; `std` is the spread across seeds, which describes stability and is **not** a confidence interval.

In [ ]:
# Features are standardised because IMV reads probabilities, and unscaled inputs
# leave logistic regression unconverged, which would score a half-optimised model.
def scaled(frame, features):
    out = frame.copy()
    out[features] = StandardScaler().fit_transform(out[features])
    return out

# Standardisation is fit once on the full dataset: with no row subsampling the
# seed no longer selects rows, only the fold partition and the estimator.
frame = scaled(data, FEATURES)

records = []
for seed in SEEDS:
    for name, factory in model_factories(seed).items():
        evaluator = BinaryIMV(
            frame, "target", FEATURES, factory,
            split_method="stratified_kfold", n_splits=N_SPLITS,
            random_seed=seed, n_jobs=N_JOBS,
        )
        evaluator.run_evaluation()
        for feature in FEATURES:
            records.append({
                "seed": seed, "model": name, "feature": feature,
                "shap_imv": evaluator.calculate_imvshapley_value(feature),
            })
        records.append({
            "seed": seed, "model": name, "feature": "__full_model_imv__",
            "shap_imv": evaluator.all_combinations_imv[tuple(FEATURES)][0],
        })

raw = pd.DataFrame(records)
raw.to_csv(RESULTS / f"{DATASET}_shap_imv_by_seed.csv", index=False)
summary = (raw[raw.feature != "__full_model_imv__"]
           .groupby(["model", "feature"])["shap_imv"]
           .agg(["mean", "std"]).reset_index()
           .sort_values(["model", "mean"], ascending=[True, False]))
summary.to_csv(RESULTS / f"{DATASET}_shap_imv_summary.csv", index=False)
summary


## 5. Figure

A negative bar means the feature *reduced* held-out information — it does not indicate the negative class.

In [ ]:
models = list(summary["model"].unique())
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.2), sharey=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, models):
    part = summary[summary.model == name].sort_values("mean", ascending=False)
    ax.bar(part["feature"], part["mean"], yerr=part["std"].fillna(0),
           capsize=3, color=sns.color_palette("crest", len(part)))
    ax.set_title(f"{TITLE}\n({name})")
    ax.tick_params(axis="x", rotation=45)
    ax.axhline(0, color="0.4", linewidth=0.8)
axes[0].set_ylabel(f"Mean SHAP-IMV over {len(SEEDS)} seeds")
fig.tight_layout()
paths = {file_format: relative_path(path, start=ARTIFACTS)
         for file_format, path in save_figure(
             fig, FIGURES / f"{DATASET}_shap_imv"
         ).items()}
plt.show()
paths
